# MiniMind M1 — Pretrain Smoke (Google Colab)

目标：只验证 `data → tokenizer → forward/backward → optimizer → checkpoint`。这不是正式训练。

固定 MiniMind commit：`6fc918beb68a0d8c40452338df6319fe168014ba`。不要在本 notebook 中 pull 或升级 MiniMind 源码。

## 0. 在 Colab 菜单选择 GPU

选择 **Runtime → Change runtime type → GPU**，再运行下面单元。

In [ ]:
import os, platform, subprocess, sys, torch
print('os=', platform.platform())
print('python=', sys.version)
print('torch=', torch.__version__)
print('torch_cuda=', torch.version.cuda)
print('cuda_available=', torch.cuda.is_available())
assert torch.cuda.is_available(), 'STOP: Colab 没有分配 CUDA GPU'
print('gpu=', torch.cuda.get_device_name(0))
print('vram_gib=', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(['nvidia-smi'], check=True)

## 1. 克隆 RadioMind 和固定 MiniMind submodule

本单元要求 `/content/RadioMind` 不存在。若重复运行，请先重启 Colab runtime，避免混入旧文件。

In [ ]:
from pathlib import Path
repo = Path('/content/RadioMind')
assert not repo.exists(), 'STOP: /content/RadioMind 已存在，请重启 runtime 后从头执行'
subprocess.run([
    'git', 'clone', '--recurse-submodules',
    'https://github.com/Gf0205/RadioMind-T1.git', str(repo)
], check=True)
main_commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
mini_commit = subprocess.check_output(['git', '-C', str(repo / 'third_party/minimind'), 'rev-parse', 'HEAD'], text=True).strip()
print('radiomind_commit=', main_commit)
print('minimind_commit=', mini_commit)
assert mini_commit == '6fc918beb68a0d8c40452338df6319fe168014ba'

## 2. 安装最小训练依赖

保留 Colab 自带 CUDA Torch，不执行 MiniMind 的完整 requirements，也不覆盖 Torch。

In [ ]:
%pip install -q 'transformers==4.57.6' 'datasets==3.6.0'

In [ ]:
import datasets, transformers
print('torch=', torch.__version__)
print('transformers=', transformers.__version__)
print('datasets=', datasets.__version__)
assert transformers.__version__ == '4.57.6'
assert datasets.__version__ == '3.6.0'
assert torch.cuda.is_available()

## 3. 从官方数据文件流式截取 256 条 smoke 样本

不会下载 1.24 GB 完整 mini 数据集；连接在取得 256 个合法 JSONL 样本后关闭。

In [ ]:
import hashlib, json, requests
source_url = 'https://huggingface.co/datasets/jingyaogong/minimind_dataset/resolve/main/pretrain_t2t_mini.jsonl'
smoke_data = repo / 'third_party/minimind/dataset/pretrain_m1_smoke_256.jsonl'
count = 0
with requests.get(source_url, stream=True, timeout=120) as response:
    response.raise_for_status()
    with smoke_data.open('w', encoding='utf-8', newline='\n') as out:
        for raw in response.iter_lines(decode_unicode=True):
            if not raw:
                continue
            row = json.loads(raw)
            assert isinstance(row.get('text'), str) and row['text']
            out.write(json.dumps(row, ensure_ascii=False) + '\n')
            count += 1
            if count == 256:
                break
assert count == 256
digest = hashlib.sha256(smoke_data.read_bytes()).hexdigest()
print('samples=', count)
print('bytes=', smoke_data.stat().st_size)
print('sha256=', digest)

## 4. Dataset/tokenizer 合同检查

In [ ]:
mini = repo / 'third_party/minimind'
sys.path.insert(0, str(mini))
from transformers import AutoTokenizer
from dataset.lm_dataset import PretrainDataset
tokenizer = AutoTokenizer.from_pretrained(mini / 'model')
dataset = PretrainDataset(str(smoke_data), tokenizer, max_length=128)
x, y = dataset[0]
print('dataset_len=', len(dataset))
print('input_shape=', tuple(x.shape), 'label_shape=', tuple(y.shape))
print('input_dtype=', x.dtype, 'label_dtype=', y.dtype)
print('bos=', int(x[0]), 'expected=', tokenizer.bos_token_id)
print('supervised_tokens=', int((y != -100).sum()))
assert len(dataset) == 256
assert tuple(x.shape) == (128,) and tuple(y.shape) == (128,)
assert x.dtype == torch.long and y.dtype == torch.long
assert int(x[0]) == tokenizer.bos_token_id

## 5. 执行受限 Pretrain smoke

固定 256 样本、1 epoch、32 micro-batches、每 4 个 micro-batch 更新一次，总计约 8 次 optimizer update。使用 FP16 兼容 T4/L4。

In [ ]:
import threading, time
trainer_dir = mini / 'trainer'
log_path = repo / 'minimind_m1_pretrain_smoke.log'
cmd = [
    sys.executable, 'train_pretrain.py',
    '--data_path', '../dataset/pretrain_m1_smoke_256.jsonl',
    '--save_dir', '../out_m1_smoke',
    '--save_weight', 'm1_smoke_pretrain',
    '--epochs', '1',
    '--batch_size', '8',
    '--accumulation_steps', '4',
    '--max_seq_len', '128',
    '--dtype', 'float16',
    '--num_workers', '2',
    '--log_interval', '1',
    '--save_interval', '16',
    '--learning_rate', '5e-4',
    '--seed', '20260907',
    '--device', 'cuda:0',
    '--from_weight', 'none',
    '--from_resume', '0',
    '--use_compile', '0',
]
print('COMMAND:', ' '.join(cmd))
memory_samples = []
monitor_stop = threading.Event()
def monitor_gpu_memory():
    while not monitor_stop.is_set():
        query = subprocess.run([
            'nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'
        ], capture_output=True, text=True)
        if query.returncode == 0:
            memory_samples.append(int(query.stdout.strip().splitlines()[0]))
        time.sleep(0.2)
process = subprocess.Popen(cmd, cwd=trainer_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
monitor = threading.Thread(target=monitor_gpu_memory, daemon=True)
monitor.start()
lines = []
for line in process.stdout:
    print(line, end='')
    lines.append(line)
returncode = process.wait()
monitor_stop.set()
monitor.join(timeout=2)
peak_used_mib = max(memory_samples) if memory_samples else None
log_path.write_text(''.join(lines), encoding='utf-8')
print('returncode=', returncode)
print('peak_gpu_memory_used_mib=', peak_used_mib)
assert returncode == 0, 'STOP: smoke training failed'

## 6. 检查 loss 与 checkpoint

In [ ]:
import math, re
log_text = log_path.read_text(encoding='utf-8')
losses = [float(v) for v in re.findall(r'loss: ([0-9.eE+-]+)', log_text)]
print('logged_losses=', len(losses))
print('first_loss=', losses[0], 'last_loss=', losses[-1])
assert losses and all(math.isfinite(v) for v in losses)
weight_path = mini / 'out_m1_smoke/m1_smoke_pretrain_768.pth'
resume_path = mini / 'checkpoints/m1_smoke_pretrain_768_resume.pth'
for path in (weight_path, resume_path, log_path):
    print(path, path.stat().st_size, 'bytes')
    assert path.is_file() and path.stat().st_size > 0
weights = torch.load(weight_path, map_location='cpu', weights_only=True)
resume = torch.load(resume_path, map_location='cpu', weights_only=False)
print('weight_tensors=', len(weights))
print('resume_epoch=', resume['epoch'], 'resume_step=', resume['step'])
print('resume_keys=', sorted(resume))
assert {'model', 'optimizer', 'epoch', 'step', 'world_size'}.issubset(resume)
assert resume['epoch'] == 0 and resume['step'] == 32
print('M1_PRETRAIN_SMOKE=PASS')

## 7. 可选：归档到 Google Drive

只在上面显示 `M1_PRETRAIN_SMOKE=PASS` 后运行。Drive 中保存权重、resume checkpoint、日志和本次 256 条数据合同。

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
archive = Path('/content/drive/MyDrive/RadioMind/minimind_m1_pretrain_smoke')
archive.mkdir(parents=True, exist_ok=True)
for path in (weight_path, resume_path, log_path, smoke_data):
    shutil.copy2(path, archive / path.name)
print('archived_to=', archive)
for path in sorted(archive.iterdir()):
    print(path.name, path.stat().st_size)